# Solution to p08

IoU is computed by broadcasting A against B, clamping negative intersection side lengths to zero. The grid target uses x with width/column and y with height/row; internal-boundary centers naturally move right/below through `floor`.

The required future multi-box generalization is deterministic: retain one source box per cell by maximizing `(area, -source_index)`. Equivalently, process boxes by decreasing area and then increasing original index, filling each cell only once. Thus a larger box wins, and equal areas preserve the lower source index.

In [ ]:
import math
import torch

boxes_a = torch.tensor([[0.,0.,6.,4.], [3.,1.,9.,7.]], dtype=torch.float32)
boxes_b = torch.tensor([[2.,1.,8.,5.]], dtype=torch.float32)
target_box = torch.tensor([2.,1.,8.,5.], dtype=torch.float32)
target_class_id = 1
image_height = 8
image_width = 12
grid_size = 4
ATOL = 1e-6
RTOL = 1e-6

def pairwise_iou(boxes_a, boxes_b):
    if not isinstance(boxes_a, torch.Tensor) or not isinstance(boxes_b, torch.Tensor):
        raise ValueError("boxes must be tensors")
    if boxes_a.ndim != 2 or boxes_b.ndim != 2 or boxes_a.shape[1:] != (4,) or boxes_b.shape[1:] != (4,):
        raise ValueError("boxes must have shape (M,4) and (N,4)")
    if boxes_a.dtype != torch.float32 or boxes_b.dtype != torch.float32:
        raise ValueError("boxes must be float32")
    if boxes_a.device.type != "cpu" or boxes_b.device.type != "cpu":
        raise ValueError("boxes must be on CPU")
    if torch.any(boxes_a[:,2:] <= boxes_a[:,:2]) or torch.any(boxes_b[:,2:] <= boxes_b[:,:2]):
        raise ValueError("boxes must have positive half-open area")
    left_top = torch.maximum(boxes_a[:,None,:2], boxes_b[None,:,:2])
    right_bottom = torch.minimum(boxes_a[:,None,2:], boxes_b[None,:,2:])
    intersection = (right_bottom - left_top).clamp_min(0).prod(dim=-1)
    area_a = (boxes_a[:,2:] - boxes_a[:,:2]).prod(dim=-1)
    area_b = (boxes_b[:,2:] - boxes_b[:,:2]).prod(dim=-1)
    union = area_a[:,None] + area_b[None,:] - intersection
    return intersection / union

def encode_grid_target(box, class_id, image_height, image_width, grid_size):
    if not isinstance(box, torch.Tensor) or box.shape != (4,) or box.dtype != torch.float32:
        raise ValueError("box must be float32 shape (4,)")
    if box.device.type != "cpu" or not isinstance(class_id, int) or class_id < 0:
        raise ValueError("invalid device or class ID")
    if min(image_height, image_width, grid_size) <= 0:
        raise ValueError("geometry must be positive")
    x0, y0, x1, y1 = (float(v) for v in box)
    if not (0 <= x0 < x1 <= image_width and 0 <= y0 < y1 <= image_height):
        raise ValueError("box must lie within the image and have positive area")
    cx, cy = (x0+x1)/2, (y0+y1)/2
    if not (0 <= cx < image_width and 0 <= cy < image_height):
        raise ValueError("center must not equal an outer boundary")
    scaled_x, scaled_y = cx*grid_size/image_width, cy*grid_size/image_height
    j, i = math.floor(scaled_x), math.floor(scaled_y)
    objectness = torch.zeros((grid_size,grid_size), dtype=torch.float32)
    classes = torch.full((grid_size,grid_size), -1, dtype=torch.int64)
    encoded_boxes = torch.zeros((grid_size,grid_size,4), dtype=torch.float32)
    objectness[i,j] = 1.0
    classes[i,j] = class_id
    encoded_boxes[i,j] = torch.tensor([
        scaled_x-j, scaled_y-i, (x1-x0)/image_width, (y1-y0)/image_height
    ], dtype=torch.float32)
    return objectness, classes, encoded_boxes

ious = pairwise_iou(boxes_a, boxes_b)
objectness, classes, encoded_boxes = encode_grid_target(
    target_box, target_class_id, image_height, image_width, grid_size
)


### Answer check

In [ ]:
torch.testing.assert_close(ious[:,0], torch.tensor([1/3,1/2]), atol=ATOL, rtol=RTOL)
assert ious.shape == (2,1) and ious.dtype == torch.float32
assert objectness.shape == (4,4) and objectness.dtype == torch.float32
assert classes.shape == (4,4) and classes.dtype == torch.int64
assert encoded_boxes.shape == (4,4,4) and encoded_boxes.dtype == torch.float32
assert torch.nonzero(objectness, as_tuple=False).tolist() == [[1,1]]
assert classes[1,1].item() == 1 and torch.all(classes[objectness == 0] == -1)
torch.testing.assert_close(encoded_boxes[1,1], torch.tensor([2/3,1/2,1/2,1/2]), atol=ATOL, rtol=RTOL)
